### 1. Import Required Libraries

In [4]:
import numpy as np
from PIL import Image  # to process image to predict it.

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
from torchvision import datasets, transforms # to use dataset from torch itself MNIST ...

from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix, classification_report

import random
import os

### 2. Configuration Parameters

In [ ]:
DATA_ROOT   = './data'
NUM_CLASSES = 10
IMG_SIZE    = 32 
IN_CHANNELS = 1      # MNIST is grayscale. 
BATCH_SIZE  = 64
EPOCHS      = 5
LR          = 1e-3
SEED        = 42
CHECKPOINT  = 'checkpoints/best_model.pth'

# Set environment for reproducibility and hardware acceleration
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs('checkpoints', exist_ok=True)
print(f"Device: {device}")

### 3. Data Preparation

#### 3.1 transform image

In [6]:
transform = transforms.Compose([               #🔴 OPTION 1 — CIFAR10 / RGB Images
    transforms.Resize((IMG_SIZE, IMG_SIZE)),  #if already resized we will not use this line
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])
 
# transforms.Normalize((0.1307,), (0.3081,)) # 🔴OPTION 2 — MNIST / GRAYSCALE Images

# transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))  #🔴 OPTION 3 — SIMPLE CUSTOM RGB DATASET (SAFE DEFAULT)

# transforms.Normalize((0.5,), (0.5,))   # 🔴 OPTION 4 — CUSTOM GRAYSCALE DATASET

IndentationError: unexpected indent (1828621717.py, line 11)

#### 3.1 Dataset Loading Stage

In [ ]:
# if you have improted dataset we will use it shortly  datasets.MNIST or datasets.CIFAR10
# if it not imported we willl use this way   torchvision.datasets.CIFAR10 to access direclty

# and also in loading  we will use DataLoader only if it imported already 
# and if not not download we will use  torch.utils.data.DataLoader to access direclty

# and num_work mean 2 workers loading data in parallel cpu use for our data.



# Load datasets

train_dataset = datasets.MNIST(root=DATA_ROOT, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root=DATA_ROOT, train=False, download=True, transform=transform)
# train_data = torchvision.datasets.CIFAR10(root = DATA_ROOT, train= True, transform=transform, download = True)
# test_data = torchvision.datasets.CIFAR10(root=DATA_ROOT,    train=False, transform=transform, download=True)


# Data Loaders

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)
# train_loader = torch.utils.data.DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_worker=2)
# test_loader = torch.utils.data.DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=True, num_worker=2)

### 4. Data Visualization (to see our data )

In [ ]:
# if the image is cifa10 we will open those two commented lines and remove the line below it 
 # image = image * 0.5 + 0.5     and  # plt.imshow(image.permute(1, 2, 0))  to change image 3 ,32,32  to 32,32,3 and squzee line will remove it work other for this
# for mnist we use squeeze to remove 1,28,28 to 28,28 


def show_sample(dataset, n=20):
    plt.figure(figsize=(10, 4))                      # Creates a figure window (size = width 10, height 4)
    for i in range(n):
        image, label = dataset[i]
        plt.subplot(2, n // 2, i + 1)
        
        # image = image * 0.5 + 0.5           
        # plt.imshow(image.permute(1, 2, 0))
     
        plt.imshow(image.squeeze(), cmap='gray')      # squeeze() removes extra channel dimension (e.g. [1,28,28] → [28,28])   # cmap='gray' means grayscale display
        plt.title(label)
        plt.axis('off')                                 # Remove x and y axis for clean look
    plt.suptitle("Sample MNIST Images", fontsize=16)
    plt.tight_layout()                                 # Automatically adjusts spacing between images
    plt.show()

show_sample(train_dataset, n=20) 

### 5. Exploratory Data Analysis (EDA) (to check data balance )

In [ ]:
# the same for all of them but for cifar we have to use class insteade of numbers so we will have to change those 
        # class_names = ['plane', 'car', 'bird', 'cat', 'deer','dog', 'frog', 'horse', 'ship', 'truck']
# and replace this line with this
        # sns.countplot(
        #     x=[class_names[label] for label in labels],
        #     palette='viridis',
        #     hue=[class_names[label] for label in labels],
        #     legend=False
        # )

# Extract all labels from the dataset
labels = [label for _, label in train_dataset]

# Create a count plot
plt.figure(figsize=(10, 5))
sns.countplot(x=labels, palette='viridis', hue=labels , legend=False)
plt.title("Class Distribution in MNIST Train Set")
plt.xlabel("Digit Class")
plt.ylabel("Frequency")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

### 6. Model Architecture and Initialization

In [ ]:
# both lines below this help us to know size of the image for our next jorney.
image,label = train_data[0]
image.size()

In [7]:
class_name = ['plane','car','bird','cat','deer','dog','frog','horse','ship','truck']
class CNNModel(nn.Module):
    def__init__(self):
        super().__init()
        self.conv1 = nn.Conv2d(3,12,5) #(12,28,28)
        self.pool = nn.MaxPool2d(2,2) # 12,14,14)
        self.conv2 = nn.Conv2d(12,24,5) # (24,10,10) ------->flatten (24,5,5)
        self.fc1 = nn.Linear(24*5*5,120)
        self.fc2 = nn.Linear(120,84)
        self.fc3 = nn.Linear(84,10)
    def forward (self,x):
        x = self.pool(F.relu(self.conv1(x))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x,1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x 
model = CNNModel().to(device)

SyntaxError: invalid syntax (619373545.py, line 3)

### 7. Loss Function and Optimizer

In [ ]:
#if the file is MNIST 
# optimizer = optim.Adam(model.parameters(), lr=LR)    fro mnist

loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(),lr=0.001,momentum=0.9) # for clasifier.

### 8. Training Loop Implementation

In [ ]:
train_losses = [] 

model.train() 
for epoch in range(EPOCHS):
    running_loss = 0.0
    
   for i, data in enumerate(train_loader):
       
        images, labels = data
        images, labels = inputs.to(device), labels.to(device)
       
        optimizer.zero_grad()
        outputs = net(images)
        loss = criterion(outputs, labels)
       
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    avg_loss = running_loss / len(train_loader) # average of all baches loss so avg_loss mean one loss of one epoch
    train_losses.append(avg_loss) 
    
    print(f"Epoch [{epoch + 1} / {EPOCHS}], Loss: {avg_loss:.4f}")

print("Training finished!")

### 9. Plot Training Loss

In [ ]:
# ... Plot Training Loss ...
plt.plot(train_losses)
plt.title("Training Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

### 10. Evaluation

In [ ]:
model.eval()
all_preds = []
all_labels = []
correct = 0
total = len(all_labels)

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.numpy())
        all_labels.extend(labels.numpy())
# for accuracy
        correct += (predicted == labels).sum().item()
        accuracy = 100 * correct / total
print(f'Accuracy: {accuracy}%')        

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

In [ ]:
# Confusion Matrix
conf_matrix = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap="Blues", xticklabels=range(10), yticklabels=range(10))
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

### 11. Save Model

In [ ]:
torch.save(model.state_dict(),'trained_net.pth')

In [ ]:
#load the trained model
model = CNNModel()
model.load_state_dict(torch.load('trained_net.pth'))

### 12. Inference on One Image

In [ ]:
# if it is MNIST
sample_img, sample_label = test_dataset[0]
plt.imshow(sample_img.squeeze(), cmap='gray')
plt.title(f"True Label: {sample_label}")
plt.axis('off')
plt.show()

sample_tensor = sample_img.unsqueeze(0)
with torch.no_grad():
    output = model(sample_tensor)
    predicted_class = torch.argmax(output).item()
    print(f"Predicted Class: {predicted_class}")



# if it is CFIAR10 and we are using image from ourself not from test 
new_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

def load_image(image_path):
    image = Image.open(image_path)
    image = new_transform(image)
    image = image.unsqueeze(0)
    return image

image_paths = ['example1.jpg', 'example2.jpg']
images = [load_image(img) for img in image_paths]

model.eval()
with torch.no_grad():
    for image in images:
        output = model(image)
        _, predicted = torch.max(output, 1)
        print(f'Predicted : {class_names[predicted.item()]}')